# Notebook 9 — Additional experiments (no retraining)

1. False alarms per normal edit at a fixed threshold (flagged area > 1% of pixels).
2. Geometric attacks (crop, rotate, flip) with a small search at verification time, for all methods.
3. Identity-bound verification (registry stores the face identity) against the copy attack.
Same inputs as notebook 8. About 1.5–2 hours.

Compares our adaptive and fixed watermarks with five published methods that have public pretrained weights:
**HiDDeN, MBRS, CIN, TrustMark and WAM (Watermark Anything)**, loaded through Meta's Video Seal code.
Every baseline is tested twice: at its authors' default strength, and re-scaled to the same PSNR as ours.

**Kaggle settings:** Input: `wm-results` (v1 files) + any of `wm_*_v2.pt` / `wm_*_v21.pt` + CelebA-HQ ·
Accelerator: **GPU** · Internet: **On**. Set `RUN_BASELINES = True` to re-run the five published methods too.
**Output:** `add_summary.md`, `add_*.csv`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lpips", "onnx", "onnxruntime-gpu", "huggingface_hub",
                "omegaconf", "einops", "pytorch_msssim", "PyWavelets", "av", "ffmpeg-python"], check=True)
try:
    import diffusers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "accelerate"], check=True)

import os, glob, time, random, urllib.request, zipfile, shutil
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm
import onnx
from onnx import numpy_helper
import onnxruntime as ort
from PIL import Image
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# ---------------- settings ----------------
N_EVAL = 1000          # same faces as notebook 4 (first 200 = calibration)
N_CAL = 200
N_UNMARKED = 500
N_DONOR = 500
N_MANIP = 200          # faces that get a face swap and a face splice
N_DIFF = 200           # faces that get a diffusion edit
DO_DIFFUSION = False   # not needed here
RUN_BASELINES = True   # the five published methods, re-matched to the main model's PSNR
MAIN_KEY = "fixed_v21" # main model
MATCH_ONLY = True      # baselines only at matched PSNR (their default-strength results are already in notebook 6)
N_GEO = 100            # faces for the geometric attacks (search is 30x slower)
N_ID = 200             # faces for the identity / copy-attack test
OUTDIR = "/kaggle/working"
MODEL_DIR = f"{OUTDIR}/models"
SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.RandomState(SEED); random.seed(SEED); torch.manual_seed(SEED)
print("Device:", device)

## Step 1 — Watermark network definitions (identical to notebook 3)

In [ ]:
def norm_layer(c, norm):
    # "gn" = group normalisation: each image is normalised on its own, so checking one photo
    # behaves exactly like training (no dependence on the rest of the batch)
    return nn.GroupNorm(min(8, c), c) if norm == "gn" else nn.BatchNorm2d(c)

def block(cin, cout, norm="bn"):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), norm_layer(cout, norm), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), norm_layer(cout, norm), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, cin, cout, ch=(32, 64, 128, 256), norm="bn"):
        super().__init__()
        self.downs = nn.ModuleList()
        c = cin
        for co in ch:
            self.downs.append(block(c, co, norm)); c = co
        self.mid = block(c, c * 2, norm); c = c * 2
        self.bottleneck_ch = c
        self.ups, self.decs = nn.ModuleList(), nn.ModuleList()
        for co in reversed(ch):
            self.ups.append(nn.ConvTranspose2d(c, co, 2, stride=2))
            self.decs.append(block(co * 2, co, norm)); c = co
        self.out = nn.Conv2d(c, cout, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x); skips.append(x); x = F.max_pool2d(x, 2)
        x = self.mid(x); bott = x
        for up, dec, s in zip(self.ups, self.decs, reversed(skips)):
            x = dec(torch.cat([up(x), s], 1))
        return self.out(x), bott

class Encoder(nn.Module):
    def __init__(self, bits):
        super().__init__()
        self.msg_fc = nn.Linear(bits, 4 * 32 * 32)
        self.unet = UNet(3 + 4, 3)
    def forward(self, x, m):
        mm = self.msg_fc(m * 2 - 1).view(-1, 4, 32, 32)
        mm = F.interpolate(mm, size=x.shape[-2:], mode="nearest")
        r, _ = self.unet(torch.cat([x, mm], 1))
        return torch.tanh(r)

class BitReader(nn.Module):
    """Separate small network that reads the hidden code (does not compete with the tamper map)."""
    def __init__(self, bits, norm="bn"):
        super().__init__()
        layers, c = [], 3
        for co in (32, 32, 64, 64, 128):                 # 256 -> 8 (stride-2 convs)
            layers += [nn.Conv2d(c, co, 3, stride=2, padding=1), norm_layer(co, norm), nn.ReLU(inplace=True)]
            c = co
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(8)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(c * 64, 512), nn.ReLU(inplace=True), nn.Linear(512, bits))
    def forward(self, x):
        return self.fc(self.pool(self.convs(x)))

class Decoder(nn.Module):
    def __init__(self, bits, norm="gn"):
        super().__init__()
        self.unet = UNet(3, 1, norm=norm)                # tamper map
        self.reader = BitReader(bits, norm=norm)         # hidden code
    def forward(self, x):
        mask, _ = self.unet(x)
        return self.reader(x), mask                      # code logits, tamper-map logits

def texture_map(x):
    g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).unsqueeze(1)
    mu = F.avg_pool2d(g, 7, 1, 3, count_include_pad=False)
    var = (F.avg_pool2d(g * g, 7, 1, 3, count_include_pad=False) - mu * mu).clamp_min(0)
    t = F.avg_pool2d(var.sqrt(), 9, 1, 4, count_include_pad=False)
    q = torch.quantile(t.flatten(1), 0.95, dim=1).view(-1, 1, 1, 1).clamp_min(1e-4)
    return (t / q).clamp(0, 1)

class WM:
    """One trained watermark model (encoder + decoder + its settings)."""
    def __init__(self, path):
        ck = torch.load(path, map_location=device, weights_only=False)
        self.mode, self.bits, self.beta, self.s_min = ck["mode"], ck["bits"], ck["beta"], ck["s_min"]
        self.dec_norm = ck.get("dec_norm", "bn")
        self.enc, self.dec = Encoder(self.bits).to(device).eval(), Decoder(self.bits, norm=self.dec_norm).to(device).eval()
        self.enc.load_state_dict(ck["enc"]); self.dec.load_state_dict(ck["dec"])
        print(f"Loaded {self.mode} model ({self.dec_norm}) from epoch {ck['epoch']} ({path})")

    def strength(self, x):
        if self.mode == "fixed":
            return torch.ones_like(x[:, :1])
        return self.s_min + (1 - self.s_min) * texture_map(x)

    @torch.no_grad()
    def embed(self, imgs, codes, bs=32):
        """uint8 RGB (N,H,W,3) + codes (N,bits) -> watermarked uint8 RGB, strength maps."""
        out, smaps = [], []
        for i in range(0, len(imgs), bs):
            x = to_t(imgs[i:i + bs])
            m = torch.from_numpy(codes[i:i + bs]).float().to(device)
            s = self.strength(x)
            xw = (x + self.beta * s * self.enc(x, m)).clamp(-1, 1)
            out.append(to_u8(xw)); smaps.append(s[:, 0].cpu().numpy())
        return np.concatenate(out), np.concatenate(smaps)

    @torch.no_grad()
    def decode(self, imgs, bs=32):
        """uint8 RGB -> decoded bits (N,bits) and tamper probability maps (N,H,W)."""
        bits, maps = [], []
        for i in range(0, len(imgs), bs):
            b, mk = self.dec(to_t(imgs[i:i + bs]))
            bits.append((b > 0).cpu().numpy().astype(np.uint8))
            maps.append(torch.sigmoid(mk[:, 0]).cpu().numpy().astype(np.float16))
        return np.concatenate(bits), np.concatenate(maps)

def to_t(imgs):
    return torch.from_numpy(np.ascontiguousarray(imgs)).to(device).permute(0, 3, 1, 2).float() / 127.5 - 1

def to_u8(x):
    return ((x.clamp(-1, 1) + 1) * 127.5).round().byte().permute(0, 2, 3, 1).cpu().numpy()

# training-style synthetic manipulation (used ONLY to set the decision threshold)
def random_region_mask(B, H, W):
    yy, xx = torch.meshgrid(torch.arange(H, device=device, dtype=torch.float32),
                            torch.arange(W, device=device, dtype=torch.float32), indexing="ij")
    ds = []
    for _ in range(B):
        if random.random() < 0.7:
            cx, cy = 128 + random.uniform(-15, 15), 135 + random.uniform(-15, 15)
            rx = random.uniform(50, 85); ry = rx * random.uniform(1.1, 1.35)
        else:
            rx = random.uniform(25, 60); ry = rx * random.uniform(0.7, 1.4)
            cx, cy = random.uniform(rx, W - rx), random.uniform(min(ry, H / 2), max(H - ry, H / 2))
        ds.append(((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2)
    return torch.sigmoid((1 - torch.stack(ds).unsqueeze(1)) * 8)

def synthetic_manipulate(xw_u8, x_u8):
    xw, x = to_t(xw_u8), to_t(x_u8)
    soft = random_region_mask(len(xw), xw.shape[2], xw.shape[3])
    src = torch.roll(x, 1, 0)
    return to_u8(xw * (1 - soft) + src * soft)

In [ ]:
# Check each image on its own statistics (fixes false alarms after JPEG / resizing)
@torch.no_grad()
def _decode_per_image(self, imgs, bs=None):
    bns = [m for m in self.dec.modules() if isinstance(m, nn.BatchNorm2d)]
    for m in bns:
        m.train(); m.momentum = 0.0      # use this image's own statistics; stored averages stay unchanged
    bits, maps = [], []
    for a in imgs:
        b, mk = self.dec(to_t(a[None]))
        bits.append((b > 0).cpu().numpy().astype(np.uint8))
        maps.append(torch.sigmoid(mk[:, 0]).cpu().numpy().astype(np.float16))
    for m in bns:
        m.eval(); m.momentum = 0.1
    return np.concatenate(bits), np.concatenate(maps)

_decode_batch = WM.decode
def _decode_auto(self, imgs, bs=32):
    # v1 (batch norm): per-image statistics; v2 (group norm): normal checking, nothing special needed
    return _decode_per_image(self, imgs) if self.dec_norm == "bn" else _decode_batch(self, imgs, bs)
WM.decode = _decode_auto
print("Per-image checking enabled for v1 models; v2 models use normal checking")

## Step 2 — Face tools: detection, deep face swap (InSwapper), classical splice

In [ ]:
URLS = {
    "yunet.onnx": "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
    "inswapper_128.onnx": "https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/inswapper_128.onnx",
    "buffalo_l.zip": "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip",
}
ARCFACE_DST = np.array([[38.2946, 51.6963], [73.5318, 51.5014], [56.0252, 71.7366],
                        [41.5493, 92.3655], [70.7299, 92.2041]], dtype=np.float32)

def fetch(name, url, tries=4):
    """Download with resume + completeness check (large files sometimes get cut off)."""
    p = os.path.join(MODEL_DIR, name)
    for _ in range(tries):
        req = urllib.request.Request(url, method="HEAD")
        total = int(urllib.request.urlopen(req).headers.get("Content-Length", 0))
        have = os.path.getsize(p) if os.path.exists(p) else 0
        if total and have == total:
            return p
        req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
        with urllib.request.urlopen(req) as r, open(p, "ab" if have else "wb") as f:
            shutil.copyfileobj(r, f, 1 << 20)
        if not total:
            return p
    raise RuntimeError(f"Could not download {name}")

def download_models():
    os.makedirs(MODEL_DIR, exist_ok=True)
    for name, url in URLS.items():
        print("model:", name); fetch(name, url)
    rec = os.path.join(MODEL_DIR, "w600k_r50.onnx")
    if not os.path.exists(rec):
        with zipfile.ZipFile(os.path.join(MODEL_DIR, "buffalo_l.zip")) as z:
            member = [m for m in z.namelist() if m.endswith("w600k_r50.onnx")][0]
            with open(rec, "wb") as f:
                f.write(z.read(member))

def _session(path):
    prov = [p for p in ["CUDAExecutionProvider", "CPUExecutionProvider"] if p in ort.get_available_providers()]
    return ort.InferenceSession(path, providers=prov)

class FaceTools:
    """Images are BGR uint8."""
    def __init__(self):
        download_models()
        self.det = cv2.FaceDetectorYN.create(os.path.join(MODEL_DIR, "yunet.onnx"), "", (320, 320), 0.6, 0.3, 5000)
        self.rec = _session(os.path.join(MODEL_DIR, "w600k_r50.onnx"))
        sw_path = os.path.join(MODEL_DIR, "inswapper_128.onnx")
        self.swap = _session(sw_path)
        self.emap = numpy_helper.to_array(onnx.load(sw_path).graph.initializer[-1])
        self.sw_in = [i.name for i in self.swap.get_inputs()]
        print("Face-swap model running on:", self.swap.get_providers()[0])

    def detect(self, img):
        h, w = img.shape[:2]
        self.det.setInputSize((w, h))
        _, faces = self.det.detect(img)
        if faces is None or len(faces) == 0:
            return None
        f = max(faces, key=lambda r: r[2] * r[3])
        lm = f[4:14].reshape(5, 2).astype(np.float32)
        eyes = lm[:2][np.argsort(lm[:2, 0])]
        mouth = lm[3:5][np.argsort(lm[3:5, 0])]
        return f[:4].astype(np.float32), np.vstack([eyes, lm[2:3], mouth])

    def crop(self, img, size=256, margin=1.3):
        """Same face crop as notebook 1 (input format of the passive detectors)."""
        d = self.detect(img)
        if d is None:
            return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
        h, w = img.shape[:2]
        x, y, bw, bh = d[0]
        cx, cy, side = x + bw / 2, y + bh / 2, max(bw, bh) * margin
        x1, y1 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
        x2, y2 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
        return cv2.resize(img[y1:y2, x1:x2], (size, size), interpolation=cv2.INTER_AREA)

    @staticmethod
    def _align(img, lm, size):
        if size % 112 == 0:
            dst = ARCFACE_DST * (size / 112.0)
        else:
            dst = ARCFACE_DST * (size / 128.0); dst[:, 0] += 8.0 * (size / 128.0)
        M, _ = cv2.estimateAffinePartial2D(lm, dst, method=cv2.LMEDS)
        return cv2.warpAffine(img, M, (size, size), borderValue=0.0), M

    def identity(self, img, lm):
        a, _ = self._align(img, lm, 112)
        blob = ((a[:, :, ::-1].astype(np.float32) - 127.5) / 127.5).transpose(2, 0, 1)[None]
        e = self.rec.run(None, {self.rec.get_inputs()[0].name: blob})[0][0]
        return e / np.linalg.norm(e)

    def inswap(self, target, t_lm, src_emb):
        aimg, M = self._align(target, t_lm, 128)
        blob = (aimg[:, :, ::-1].astype(np.float32) / 255.0).transpose(2, 0, 1)[None]
        lat = src_emb.reshape(1, -1) @ self.emap
        lat = (lat / np.linalg.norm(lat)).astype(np.float32)
        pred = self.swap.run(None, {self.sw_in[0]: blob, self.sw_in[1]: lat})[0][0]
        fake = (pred.transpose(1, 2, 0).clip(0, 1) * 255)[:, :, ::-1].astype(np.float32)
        h, w = target.shape[:2]
        IM = cv2.invertAffineTransform(M)
        fake_w = cv2.warpAffine(fake, IM, (w, h), borderValue=0.0)
        mask = cv2.warpAffine(np.full((128, 128), 255, np.float32), IM, (w, h), borderValue=0.0)
        mask[mask > 20] = 255
        ys, xs = np.where(mask == 255)
        msize = int(np.sqrt((ys.max() - ys.min()) * (xs.max() - xs.min())))
        k = max(msize // 10, 10)
        mask = cv2.erode(mask, np.ones((k, k), np.uint8), iterations=1)
        k = max(msize // 20, 5)
        mask = cv2.GaussianBlur(mask, (2 * k + 1, 2 * k + 1), 0) / 255.0
        out = mask[..., None] * fake_w + (1 - mask[..., None]) * target.astype(np.float32)
        return out.clip(0, 255).astype(np.uint8), (mask > 0.5).astype(np.uint8)

    def splice(self, target, t_lm, donor, d_lm):
        M, _ = cv2.estimateAffinePartial2D(d_lm, t_lm, method=cv2.LMEDS)
        h, w = target.shape[:2]
        warped = cv2.warpAffine(donor, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        eye_c, mouth_c = t_lm[:2].mean(0), t_lm[3:5].mean(0)
        eye_d = np.linalg.norm(t_lm[1] - t_lm[0])
        center = (eye_c + mouth_c) / 2
        axes = (int(eye_d * 1.05), int(np.linalg.norm(mouth_c - eye_c) * 1.25))
        ang = np.degrees(np.arctan2(*(mouth_c - eye_c)[::-1])) - 90
        mask = np.zeros((h, w), np.uint8)
        cv2.ellipse(mask, (int(center[0]), int(center[1])), axes, ang, 0, 360, 255, -1)
        ys, xs = np.where(mask > 0)
        c = (int((xs.min() + xs.max()) / 2), int((ys.min() + ys.max()) / 2))
        try:
            out = cv2.seamlessClone(warped, target, mask.copy(), c, cv2.NORMAL_CLONE)
        except cv2.error:
            m = cv2.GaussianBlur(mask, (15, 15), 0)[..., None] / 255.0
            out = (m * warped + (1 - m) * target).astype(np.uint8)
        return out, (mask > 0).astype(np.uint8)

    @staticmethod
    def edit_mask(shape, lm, part):
        h, w = shape[:2]
        eye_d = np.linalg.norm(lm[1] - lm[0])
        m = np.zeros((h, w), np.uint8)
        if part == "mouth":
            c = lm[3:5].mean(0); ax = (int(eye_d * 0.75), int(eye_d * 0.45))
        else:
            c = lm[:2].mean(0); ax = (int(eye_d * 1.05), int(eye_d * 0.4))
        cv2.ellipse(m, (int(c[0]), int(c[1])), ax, 0, 0, 360, 1, -1)
        return m

ft = FaceTools()
rgb2bgr = lambda a: np.ascontiguousarray(a[:, :, ::-1])

## Step 3 — Load models and test images

In [ ]:
wm_paths = sorted(p for p in glob.glob("/kaggle/input/**/wm_*.pt", recursive=True) if not p.endswith("_last.pt"))
WMS = {}
for p in wm_paths:
    key = os.path.basename(p)[3:-3]          # adaptive, fixed, adaptive_v2, fixed_v2
    if key not in WMS:
        WMS[key] = WM(p)
assert WMS, "no wm_*.pt files found - add the notebook 3 outputs as Input"
NAMES = {"fixed": "Ours v1 (fixed)", "adaptive": "Ours v1 (adaptive)",
         "fixed_v2": "Ours v2 (fixed)", "adaptive_v2": "Ours v2 (adaptive)",
         "fixed_v21": "Ours v2.1 (fixed)", "adaptive_v21": "Ours v2.1 (adaptive)"}
MODES = [m for m in NAMES if m in WMS and m.endswith("_v21")]   # only the final models here
skipped = [k for k in WMS if k not in NAMES]
if skipped:
    print("!! ignoring unrecognised model files:", skipped)
print("Our models:", MODES)
BITS = WMS[MODES[0]].bits

split_csv = glob.glob("/kaggle/input/**/celeba_split.csv", recursive=True)[0]
split = pd.read_csv(split_csv)
all_imgs = glob.glob("/kaggle/input/**/*.jpg", recursive=True) + glob.glob("/kaggle/input/**/*.png", recursive=True)
by_name = {os.path.basename(p): p for p in all_imgs}
test_files = [by_name[os.path.basename(p)] for p in split[split.split == "test"].path if os.path.basename(p) in by_name]
print("CelebA-HQ test images available:", len(test_files))

def load_faces(files, need):
    """Load images, keep only those where a face (with landmarks) is found."""
    imgs, lms = [], []
    for f in files:
        a = np.array(Image.open(f).convert("RGB").resize((256, 256), Image.BICUBIC))
        d = ft.detect(rgb2bgr(a))
        if d is not None:
            imgs.append(a); lms.append(d[1])
        if len(imgs) == need:
            break
    return np.stack(imgs), np.stack(lms)

X, LM = load_faces(test_files[:1200], N_EVAL)
XU, LMU = load_faces(test_files[1200:1700], N_UNMARKED)
XD, LMD = load_faces(test_files[1700:], N_DONOR)
print("eval", X.shape, "| unmarked", XU.shape, "| donors", XD.shape)
CAL, EV = np.arange(N_CAL), np.arange(N_CAL, len(X))

CODES = rng.randint(0, 2, (len(X), BITS)).astype(np.uint8)   # one secret code per published image
REG = CODES                                                  # the registry of issued codes
donor_emb = np.stack([ft.identity(rgb2bgr(XD[i]), LMD[i]) for i in tqdm(range(len(XD)), desc="donor identities")])

det_paths = sorted(glob.glob("/kaggle/input/**/det_*.pt", recursive=True))
DETS = {}
for p in det_paths:
    ck = torch.load(p, map_location=device, weights_only=False)
    m = timm.create_model(ck["timm_name"], pretrained=False, num_classes=1)
    m.load_state_dict(ck["state"]); DETS[os.path.basename(p)[4:-3]] = m.to(device).eval()
DET_NAMES = {"xception": "Xception", "effb4": "EfficientNet-B4"}
print("Passive detectors:", list(DETS))

## Step 4 — Helper functions: distortions, scoring, metrics

In [ ]:
def jpeg(img, q):
    ok, buf = cv2.imencode(".jpg", rgb2bgr(img), [cv2.IMWRITE_JPEG_QUALITY, int(q)])
    return cv2.imdecode(buf, cv2.IMREAD_COLOR)[:, :, ::-1]

def resize(img, s):
    h, w = img.shape[:2]
    small = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

_nrng = np.random.RandomState(1)
DISTORTIONS = {
    "None": lambda a: a,
    "JPEG 90": lambda a: jpeg(a, 90),
    "JPEG 70": lambda a: jpeg(a, 70),
    "JPEG 50": lambda a: jpeg(a, 50),
    "Resize 0.75": lambda a: resize(a, 0.75),
    "Resize 0.5": lambda a: resize(a, 0.5),
    "Blur (sigma 1)": lambda a: cv2.GaussianBlur(a, (5, 5), 1.0),
    "Noise (sigma 2.5)": lambda a: np.clip(a + _nrng.normal(0, 2.55, a.shape), 0, 255).astype(np.uint8),
    "Brightness +10%": lambda a: np.clip(a.astype(np.float32) * 1.1, 0, 255).astype(np.uint8),
    "Social media (resize 0.75 + JPEG 75)": lambda a: jpeg(resize(a, 0.75), 75),
}
POST = {"raw": lambda a: a, "+JPEG 75": lambda a: jpeg(a, 75)}

def apply(imgs, fn):
    return np.stack([fn(a) for a in imgs])

def random_benign(imgs, seed):
    r = np.random.RandomState(seed); keys = list(DISTORTIONS)
    return np.stack([DISTORTIONS[keys[r.randint(len(keys))]](a) for a in imgs])

def wm_scores(wm, imgs, codes=None, gts=None):
    bits, maps = wm.decode(imgs)
    pred = maps > 0.5
    s = {"tamper": pred.mean((1, 2))}
    agree = (bits.astype(np.float32) @ REG.T.astype(np.float32) +
             (1 - bits).astype(np.float32) @ (1 - REG).T.astype(np.float32)) / BITS
    s["match"] = agree.max(1)                         # best match against all registered codes
    if codes is not None:
        s["bit_acc"] = (bits == codes).mean(1)
    if gts is not None:
        g = gts.astype(bool)
        inter = (pred & g).sum((1, 2)); union = (pred | g).sum((1, 2))
        s["iou"] = inter / np.maximum(union, 1)
        s["f1"] = 2 * inter / np.maximum(pred.sum((1, 2)) + g.sum((1, 2)), 1)
    return s, maps

@torch.no_grad()
def passive_scores(model, imgs, bs=64):
    cfg = timm.data.resolve_data_config({}, model=model)
    norm = T.Normalize(cfg["mean"], cfg["std"])
    probs = []
    for i in range(0, len(imgs), bs):
        crops = [ft.crop(rgb2bgr(a))[:, :, ::-1] for a in imgs[i:i + bs]]
        x = torch.stack([norm(TF.to_tensor(c.copy())) for c in crops]).to(device)
        probs.append(torch.sigmoid(model(x).float().squeeze(1)).cpu().numpy())
    return np.concatenate(probs)

def best_threshold(neg, pos):
    cands = np.unique(np.concatenate([neg, pos, [0.0, 1.0]]))
    bal = [((neg <= t).mean() + (pos > t).mean()) / 2 for t in cands]
    return float(cands[int(np.argmax(bal))])

def auc(neg, pos):
    return roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos])

def cls_metrics(y, yhat):
    tp = ((y == 1) & (yhat == 1)).sum(); tn = ((y == 0) & (yhat == 0)).sum()
    fp = ((y == 0) & (yhat == 1)).sum(); fn = ((y == 1) & (yhat == 0)).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(ACC=(tp + tn) / len(y), Precision=prec, Recall=rec,
                F1=2 * prec * rec / max(prec + rec, 1e-9))

## Step 5 — Published baselines (HiDDeN, MBRS, CIN, TrustMark, WAM) through Meta's Video Seal wrappers

In [ ]:
import subprocess as _sp
VS_DIR = f"{OUTDIR}/videoseal"
if RUN_BASELINES and not os.path.exists(VS_DIR):
    _sp.run(["git", "clone", "-q", "--depth", "1", "https://github.com/facebookresearch/videoseal.git", VS_DIR], check=True)
CK_DIR = os.path.join(VS_DIR, "ckpts")
if RUN_BASELINES and not os.path.exists(os.path.join(CK_DIR, "wam_decoder.pt")):
    from huggingface_hub import snapshot_download
    snapshot_download("tangtianzhong/img-wm-torchscript", local_dir=CK_DIR)
if RUN_BASELINES:
    sys.path.insert(0, VS_DIR)
    from videoseal.models import build_baseline

CWD0 = os.getcwd()

class OursMethod:
    def __init__(self, wm, name):
        self.wm, self.name, self.nbits, self.has_map, self.kind = wm, name, wm.bits, True, "ours"
    def embed(self, imgs, codes):
        return self.wm.embed(imgs, codes)[0]
    def decode(self, imgs):
        bits, maps = self.wm.decode(imgs)
        return bits, maps.astype(np.float32) > 0.5            # True = mark missing (tampered)

class VSMethod:
    """A published baseline, loaded and run exactly as in Meta's Video Seal code."""
    def __init__(self, key, name):
        os.chdir(VS_DIR)
        try:
            self.m = build_baseline(key).to(device).eval()
        finally:
            os.chdir(CWD0)
        self.key, self.name, self.kind = key, name, "baseline"
        self.nbits = self.m.embedder.nbits
        self.has_map = key.startswith("wam")
        self.default_scale = float(self.m.blender.scaling_w)
        self.flip_bits, self.flip_map = False, False

    @torch.no_grad()
    def embed(self, imgs, codes, bs=16):
        out = []
        for i in range(0, len(imgs), bs):
            x = torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).permute(0, 3, 1, 2).float().div(255).to(device)
            m = torch.from_numpy(codes[i:i + bs].astype(np.int64)).to(device)
            xw = self.m.embed(x, m, is_video=False)["imgs_w"]
            out.append((xw.clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy())
        return np.concatenate(out)

    @torch.no_grad()
    def decode(self, imgs, bs=16):
        bits, maps = [], []
        for i in range(0, len(imgs), bs):
            x = torch.from_numpy(np.ascontiguousarray(imgs[i:i + bs])).permute(0, 3, 1, 2).float().div(255).to(device)
            preds = self.m.detect(x, is_video=False)["preds"]
            b = (preds[:, 1:] > 0).cpu().numpy().astype(np.uint8)
            bits.append(1 - b if self.flip_bits else b)
            if self.has_map:   # WAM: per-pixel "is watermarked" logits in channel 0
                raw = self.m.detector.decoder(self.m.detector.preprocess(x))
                p = torch.sigmoid(raw[:, 0].float())
                if p.shape[-2:] != x.shape[-2:]:
                    p = F.interpolate(p[:, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
                missing = (p > 0.5) if self.flip_map else (p < 0.5)
                maps.append(missing.cpu().numpy())
        return np.concatenate(bits), (np.concatenate(maps) if self.has_map else None)

    def set_scale(self, s):
        self.m.blender.scaling_w = float(s)

BASELINES = [("hidden", "HiDDeN"), ("mbrs", "MBRS"), ("cin", "CIN"), ("trustmark", "TrustMark"), ("wam", "WAM")]
METHODS = {}
for mode in MODES:
    METHODS[NAMES[mode]] = OursMethod(WMS[mode], NAMES[mode])
for key, name in (BASELINES if RUN_BASELINES else []):
    try:
        METHODS[name] = VSMethod(key, name)
    except Exception as e:
        print(f"!! could not load {name}: {str(e)[:200]}")

def psnr_u8(a, b):
    mse = ((a.astype(np.float32) - b.astype(np.float32)) ** 2).reshape(len(a), -1).mean(1)
    return 10 * np.log10(255 ** 2 / np.maximum(mse, 1e-10))

# sanity check on 16 calibration faces: clean bit accuracy must be high; fix sign conventions if inverted
SAN = X[CAL[:16]]
for k, meth in METHODS.items():
    codes = np.random.RandomState(5).randint(0, 2, (len(SAN), meth.nbits)).astype(np.uint8)
    xw = meth.embed(SAN, codes)
    bits, maps = meth.decode(xw)
    acc = (bits == codes).mean()
    if meth.kind == "baseline" and acc < 0.3:
        meth.flip_bits = True; acc = 1 - acc
        print(f"   note: {k} bits were inverted - corrected")
    msg = f"{k:15s} bits={meth.nbits:3d}  PSNR={psnr_u8(xw, SAN).mean():6.2f} dB  clean bit acc={acc:.4f}"
    if meth.has_map:
        _, m_un = meth.decode(SAN)
        if meth.kind == "baseline" and maps.mean() > m_un.mean():
            meth.flip_map = True
            _, maps = meth.decode(xw); _, m_un = meth.decode(SAN)
            print(f"   note: {k} mask sign was inverted - corrected")
        msg += f"  | 'mark missing' area: watermarked {maps.mean():.3f}, unmarked {m_un.mean():.3f}"
    print(msg)
    if acc < 0.9:
        print(f"   !! WARNING: {k} clean bit accuracy is low - check this method")

## Step 6 — Strength settings: authors' defaults, and a second copy of every baseline matched to our PSNR

In [ ]:
TARGET_PSNR_MATCH = None
if NAMES[MAIN_KEY] in METHODS:
    TARGET_PSNR_MATCH = float(psnr_u8(METHODS[NAMES[MAIN_KEY]].embed(X[CAL[:64]], CODES[CAL[:64]]), X[CAL[:64]]).mean())
print("PSNR to match:", TARGET_PSNR_MATCH)

def mean_psnr(meth, imgs):
    codes = np.random.RandomState(6).randint(0, 2, (len(imgs), meth.nbits)).astype(np.uint8)
    return float(psnr_u8(meth.embed(imgs, codes), imgs).mean())

CONFIGS = {}          # config name -> (method, scale or None)
for k, meth in METHODS.items():
    if meth.kind == "ours":
        CONFIGS[meth.name] = (meth, None)
        continue
    if not MATCH_ONLY:
        CONFIGS[f"{meth.name} (default)"] = (meth, meth.default_scale)
    if TARGET_PSNR_MATCH is not None:
        lo, hi = np.log(meth.default_scale / 50), np.log(meth.default_scale * 50)
        for _ in range(16):                                   # bisection on log-strength
            mid = (lo + hi) / 2
            meth.set_scale(np.exp(mid))
            if mean_psnr(meth, X[CAL[:64]]) > TARGET_PSNR_MATCH:
                lo = mid
            else:
                hi = mid
        s = float(np.exp((lo + hi) / 2)); meth.set_scale(s)
        print(f"{meth.name}: default scale {meth.default_scale:.4g} -> matched scale {s:.4g} "
              f"(PSNR {mean_psnr(meth, X[CAL[:64]]):.2f} dB)")
        CONFIGS[f"{meth.name} (matched PSNR)"] = (meth, s)
        meth.set_scale(meth.default_scale)
print("Configurations:", list(CONFIGS))

def use(cfg):
    meth, s = CONFIGS[cfg]
    if s is not None:
        meth.set_scale(s)
    return meth

## Step 7 — Manipulations generated once on the clean faces, then composited onto each watermarked version
The edited region is newly generated content in every case, so pasting it (with the tool's own blending mask)
onto each method's watermarked image gives the same manipulated image the tool would produce, at a fraction of the cost.

In [ ]:
def inswap_soft(target, t_lm, src_emb):
    aimg, M = ft._align(target, t_lm, 128)
    blob = (aimg[:, :, ::-1].astype(np.float32) / 255.0).transpose(2, 0, 1)[None]
    lat = src_emb.reshape(1, -1) @ ft.emap
    lat = (lat / np.linalg.norm(lat)).astype(np.float32)
    pred = ft.swap.run(None, {ft.sw_in[0]: blob, ft.sw_in[1]: lat})[0][0]
    fake = (pred.transpose(1, 2, 0).clip(0, 1) * 255)[:, :, ::-1].astype(np.float32)
    h, w = target.shape[:2]
    IM = cv2.invertAffineTransform(M)
    fake_w = cv2.warpAffine(fake, IM, (w, h), borderValue=0.0)
    mask = cv2.warpAffine(np.full((128, 128), 255, np.float32), IM, (w, h), borderValue=0.0)
    mask[mask > 20] = 255
    ys, xs = np.where(mask == 255)
    msize = int(np.sqrt((ys.max() - ys.min()) * (xs.max() - xs.min())))
    k = max(msize // 10, 10)
    mask = cv2.erode(mask, np.ones((k, k), np.uint8), iterations=1)
    k = max(msize // 20, 5)
    mask = cv2.GaussianBlur(mask, (2 * k + 1, 2 * k + 1), 0) / 255.0
    return fake_w[:, :, ::-1].clip(0, 255).astype(np.float32), mask.astype(np.float32)   # RGB content, soft mask

MI = EV[:N_MANIP]
MAN = {}   # name -> (content RGB float (N,H,W,3), soft mask (N,H,W), indices into X)
cont, soft = [], []
for n, i in enumerate(tqdm(MI, desc="face swaps")):
    c, s = inswap_soft(rgb2bgr(X[i]), LM[i], donor_emb[(n * 7) % len(XD)]); cont.append(c); soft.append(s)
MAN["Face swap"] = (np.stack(cont), np.stack(soft), MI)
cont, soft = [], []
for n, i in enumerate(tqdm(MI, desc="face splices")):
    j = (n * 7) % len(XD)
    out, g = ft.splice(rgb2bgr(X[i]), LM[i], rgb2bgr(XD[j]), LMD[j])
    cont.append(out[:, :, ::-1].astype(np.float32)); soft.append(g.astype(np.float32))
MAN["Face splice"] = (np.stack(cont), np.stack(soft), MI)

DI = EV[:N_DIFF]
if DO_DIFFUSION:
    try:
        from diffusers import StableDiffusionInpaintPipeline
        pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "stable-diffusion-v1-5/stable-diffusion-inpainting", torch_dtype=torch.float16,
            safety_checker=None, requires_safety_checker=False).to(device)
        pipe.set_progress_bar_config(disable=True)
        PROMPTS = {"mouth": ["a smiling mouth", "an open mouth showing teeth", "closed lips, neutral expression"],
                   "eyes": ["closed eyes", "eyes looking sideways", "wide open eyes"]}
        cont, soft = [], []
        for k0 in tqdm(range(0, len(DI), 4), desc="diffusion edits"):
            batch = DI[k0:k0 + 4]; ims, masks, prompts, sm = [], [], [], []
            for n, i in enumerate(batch):
                part = "mouth" if (k0 + n) % 2 == 0 else "eyes"
                m = ft.edit_mask(X[i].shape, LM[i], part)
                ims.append(Image.fromarray(X[i]).resize((512, 512), Image.BICUBIC))
                masks.append(Image.fromarray(cv2.dilate(m, np.ones((7, 7), np.uint8)) * 255).resize((512, 512), Image.NEAREST))
                prompts.append("photo of a person, " + PROMPTS[part][(k0 + n) % 3] + ", high quality, detailed")
                sm.append(cv2.GaussianBlur(m.astype(np.float32), (9, 9), 3))
            res = pipe(prompt=prompts, image=ims, mask_image=masks, num_inference_steps=25, guidance_scale=7.5,
                       generator=[torch.Generator(device).manual_seed(int(i)) for i in batch]).images
            for n in range(len(batch)):
                cont.append(np.array(res[n].resize((256, 256), Image.BICUBIC)).astype(np.float32)); soft.append(sm[n])
        MAN["Diffusion inpainting"] = (np.stack(cont), np.stack(soft), DI)
        del pipe; torch.cuda.empty_cache()
    except Exception as e:
        print("Diffusion attack skipped:", str(e)[:200])

def composite(xw_sub, man):
    c, s, _ = man
    return (s[..., None] * c + (1 - s[..., None]) * xw_sub.astype(np.float32)).round().clip(0, 255).astype(np.uint8)

GT = {k: v[1] > 0.5 for k, v in MAN.items()}
print({k: (len(v[2]), round(float(GT[k].mean()), 3)) for k, v in MAN.items()})

## Step 8 — Shared settings: normal edits and the random edit given to each face

In [ ]:
import lpips
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
lp = lpips.LPIPS(net="alex", verbose=False).to(device).eval()
DIST = dict(DISTORTIONS); DIST["JPEG 30"] = lambda a: jpeg(a, 30)
BENIGN_KEYS = [k for k in DIST if k != "JPEG 30"]
_r = np.random.RandomState(31)
MIX_NEG = {i: BENIGN_KEYS[_r.randint(len(BENIGN_KEYS))] for i in range(len(X))}     # same edits for every method
MIX_POS = {i: BENIGN_KEYS[_r.randint(len(BENIGN_KEYS))] for i in range(len(X))}
mixed = lambda imgs, idx, table: np.stack([DIST[table[int(i)]](a) for a, i in zip(imgs, idx)])

def tpr_at_fpr(neg, pos, fpr=0.01):
    t = np.quantile(neg, 1 - fpr)
    return float((pos > t).mean())

def quality(xw, x):
    a, b = to_t(xw), to_t(x)
    with torch.no_grad():
        ss = np.concatenate([ssim_fn((a[i:i+50]+1)/2, (b[i:i+50]+1)/2, data_range=1.0, reduction="none").cpu().numpy() for i in range(0, len(a), 50)])
        lv = np.concatenate([lp(a[i:i+50], b[i:i+50]).flatten().cpu().numpy() for i in range(0, len(a), 50)])
    return float(psnr_u8(xw, x).mean()), float(ss.mean()), float(lv.mean())

def speed(meth):
    codes = np.zeros((1, meth.nbits), np.uint8)
    for _ in range(5): meth.decode(meth.embed(X[:1], codes))
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20): xw = meth.embed(X[:1], codes)
    if device == "cuda": torch.cuda.synchronize()
    te = (time.time() - t0) / 20 * 1000; t0 = time.time()
    for _ in range(20): meth.decode(xw)
    if device == "cuda": torch.cuda.synchronize()
    return te, (time.time() - t0) / 20 * 1000

## Step 9 — Helpers: geometric search at verification, identity check

In [ ]:
from math import comb
def verify_threshold(k, registry=1000, trials=1, max_false=0.01):
    """Smallest number of matching bits so that a random image matches any registered code
    (over all search trials) with probability <= 1%."""
    for t in range(k // 2, k + 1):
        if registry * trials * sum(comb(k, j) for j in range(t, k + 1)) / 2 ** k <= max_false:
            return t
    return k

def best_match(bits, registry):
    agree = (bits.astype(np.float32) @ registry.T.astype(np.float32) +
             (1 - bits).astype(np.float32) @ (1 - registry).T.astype(np.float32))
    return agree.max(1), agree.argmax(1)

def rot(a, d, border=cv2.BORDER_REFLECT):
    h, w = a.shape[:2]; M = cv2.getRotationMatrix2D((w / 2, h / 2), d, 1.0)
    return cv2.warpAffine(a, M, (w, h), borderMode=border, borderValue=0)

def crop(a, f):
    h = a.shape[0]; c = int(h * f); o = (h - c) // 2
    return cv2.resize(a[o:o + c, o:o + c], (h, h), interpolation=cv2.INTER_LINEAR)

def uncrop(a, f, border=cv2.BORDER_REFLECT):
    h = a.shape[0]; c = int(round(h * f)); small = cv2.resize(a, (c, c), interpolation=cv2.INTER_AREA); o = (h - c) // 2
    return cv2.copyMakeBorder(small, o, h - c - o, o, h - c - o, border, value=0)

CANDS = [(fl, r, s) for fl in (0, 1) for r in (-10, -5, 0, 5, 10) for s in (1.0, 0.9, 0.75)]

def undo(a, fl, r, s, border=cv2.BORDER_REFLECT):
    b = uncrop(a, s, border) if s < 1 else a
    b = rot(b, -r, border) if r else b
    return np.ascontiguousarray(b[:, ::-1]) if fl else b

VALID = {c: undo(np.ones((256, 256), np.float32), *c, border=cv2.BORDER_CONSTANT) > 0.99 for c in CANDS}

def search_decode(meth, imgs, registry):
    """Try every candidate undo-transform; keep the one whose code best matches the registry."""
    best_s = np.full(len(imgs), -1.0); best_bits = np.zeros((len(imgs), meth.nbits), np.uint8)
    best_area = np.zeros(len(imgs)); best_c = [None] * len(imgs)
    for c in CANDS:
        al = np.stack([undo(a, *c) for a in imgs])
        b, mp = meth.decode(al)
        s, _ = best_match(b, registry)
        upd = s > best_s
        best_s[upd] = s[upd]; best_bits[upd] = b[upd]
        if mp is not None:
            v = VALID[c]
            area = (mp & v).sum((1, 2)) / v.sum()
            best_area[upd] = area[upd]
        for i in np.where(upd)[0]:
            best_c[i] = c
    return best_bits, best_s, best_area, best_c

def identity_of(imgs):
    """ArcFace identity embedding of the face in each image (None if no face is found)."""
    out = []
    for a in imgs:
        d = ft.detect(rgb2bgr(a))
        out.append(None if d is None else ft.identity(rgb2bgr(a), d[1]))
    return out

def id_sim(e1, e2):
    return np.array([-1.0 if (a is None or b is None) else float(a @ b) for a, b in zip(e1, e2)])

GEO_ATTACKS = {   # on the search grid
    "Crop 90%": lambda a: crop(a, 0.9), "Crop 75%": lambda a: crop(a, 0.75),
    "Rotate 5°": lambda a: rot(a, 5), "Rotate 10°": lambda a: rot(a, 10),
    "Horizontal flip": lambda a: np.ascontiguousarray(a[:, ::-1]),
    # off the search grid / combined with compression
    "Rotate 3° (off grid)": lambda a: rot(a, 3), "Rotate 7° (off grid)": lambda a: rot(a, 7),
    "Crop 85% (off grid)": lambda a: crop(a, 0.85),
    "Rotate 5° + JPEG 75": lambda a: jpeg(rot(a, 5), 75),
    "Flip + crop 90% + JPEG 75": lambda a: jpeg(crop(np.ascontiguousarray(a[:, ::-1]), 0.9), 75),
}

## Step 10 — Fixed operating point: flagged area > max(1% of pixels, 99th percentile on calibration faces)
Per-edit false-alarm rates (genuine images) and detection rates, for the methods that produce a tamper map.

In [ ]:
OP = []
TAU = {}
for cfg in CONFIGS:
    meth = use(cfg)
    if not meth.has_map:
        continue
    codes = np.random.RandomState(7).randint(0, 2, (len(X), meth.nbits)).astype(np.uint8)
    xw = meth.embed(X, codes)
    _, m_cal = meth.decode(mixed(xw[CAL], CAL, MIX_NEG))
    tau = float(max(np.quantile(m_cal.mean((1, 2)), 0.99), 0.01)); TAU[cfg] = tau
    row = {"Method": cfg, "Threshold (flagged area)": tau}
    for dn, fn in DIST.items():
        _, mp = meth.decode(apply(xw[EV], fn))
        row[f"False alarm: {dn}"] = float(100 * (mp.mean((1, 2)) > tau).mean())
    for mn, man in MAN.items():
        idx = man[2]; fake = composite(xw[idx], man)
        for post in ["raw", "mixed edits"]:
            im = fake if post == "raw" else mixed(fake, idx, MIX_POS)
            _, mp = meth.decode(im)
            row[f"Detected: {mn} ({post})"] = float(100 * (mp.mean((1, 2)) > tau).mean())
    OP.append(row); print(cfg, "threshold", round(tau, 4))
T_OP = pd.DataFrame(OP).set_index("Method").T.reset_index().rename(columns={"index": "Setting"})
T_OP.to_csv(f"{OUTDIR}/add_operating_point.csv", index=False)
print(T_OP.round(3).to_string(index=False))

## Step 11 — Geometric attacks with search at verification (all methods, same search)

In [ ]:
GI = EV[:N_GEO]
GEO = []
for cfg in CONFIGS:
    meth = use(cfg); k = meth.nbits
    codes = np.random.RandomState(7).randint(0, 2, (len(X), k)).astype(np.uint8)
    t1, tS = verify_threshold(k), verify_threshold(k, trials=len(CANDS))
    xw = meth.embed(X[GI], codes[GI])
    # false matches: unmarked faces, with the same search
    _, s_un, _, _ = search_decode(meth, XU[:N_GEO], codes)
    false_match = float(100 * (s_un >= tS).mean())
    for an, fn in GEO_ATTACKS.items():
        att = apply(xw, fn)
        b0, m0 = meth.decode(att)
        s0, _ = best_match(b0, codes)
        bS, sS, areaS, cS = search_decode(meth, att, codes)
        r = {"Method": cfg, "Attack": an, "Bits": k,
             "Bit acc. (no search)": float((b0 == codes[GI]).mean()),
             "Verified % (no search)": float(100 * (s0 >= t1).mean()),
             "Bit acc. (search)": float((bS == codes[GI]).mean()),
             "Verified % (search)": float(100 * (sS >= tS).mean()),
             "Unmarked faces falsely verified % (search)": false_match}
        if meth.has_map:
            tau = TAU.get(cfg, 0.01)
            r["False alarm % (no search)"] = float(100 * (m0.mean((1, 2)) > tau).mean())
            r["False alarm % (search)"] = float(100 * (areaS > tau).mean())
        GEO.append(r)
    print(f"{cfg}: thresholds {t1}/{k} (single) and {tS}/{k} (search over {len(CANDS)}); unmarked false matches {false_match:.1f}%")
T_GEO = pd.DataFrame(GEO)
T_GEO.to_csv(f"{OUTDIR}/add_geometric.csv", index=False)
print(T_GEO.round(3).to_string(index=False))

## Step 12 — Identity-bound verification against the copy attack
The registry stores, with each code, the face-identity embedding (ArcFace) of the published image.
A checked image is accepted as genuine only if its code matches AND its face matches the registered identity.

In [ ]:
II = EV[:min(N_ID, len(XU))]
TG = XU[:len(II)]
ID_ROWS = []
emb_tgt = identity_of(TG)
for cfg in CONFIGS:
    meth = use(cfg); k = meth.nbits; t1 = verify_threshold(k)
    codes = np.random.RandomState(7).randint(0, 2, (len(X), k)).astype(np.uint8)
    xw_cal = meth.embed(X[CAL], codes[CAL])
    sim_cal = id_sim(identity_of(mixed(xw_cal, CAL, MIX_NEG)), identity_of(xw_cal))
    theta = float(min(np.quantile(sim_cal[sim_cal > -1], 0.01), 0.5))   # <=1% of genuine images may fail; never above 0.5
    xw = meth.embed(X[II], codes[II]); emb_pub = identity_of(xw)      # stored at publication time
    resid = xw.astype(np.float32) - X[II].astype(np.float32)
    blind = xw.astype(np.float32) - np.stack([cv2.GaussianBlur(a, (0, 0), 1.0) for a in xw]).astype(np.float32)
    cases = {
        "Genuine (random normal edit)": mixed(xw, II, MIX_NEG),
        "Copy attack, exact watermark": np.clip(TG + resid, 0, 255).astype(np.uint8),
        "Copy attack, estimated watermark": np.clip(TG + blind, 0, 255).astype(np.uint8),
    }
    if "Face swap" in MAN:
        sw = MAN["Face swap"]; j = np.isin(sw[2], II)
        xs = meth.embed(X[sw[2][j]], codes[sw[2][j]])
        cases["Face swap (random normal edit)"] = mixed(composite(xs, (sw[0][j], sw[1][j], sw[2][j])), sw[2][j], MIX_POS)
    for cn, im in cases.items():
        b, mp = meth.decode(im)
        s, who = best_match(b, codes)
        code_ok = s >= t1
        claimed = [emb_pub[np.where(II == w)[0][0]] if w in II else None for w in who]
        if cn.startswith("Face swap"):
            claimed = identity_of(xs)
        sim = id_sim(identity_of(im), claimed)
        id_ok = sim >= theta
        row = {"Method": cfg, "Case": cn, "Identity threshold": theta,
               "Accepted as genuine: code only (%)": float(100 * code_ok.mean()),
               "Accepted as genuine: code + identity (%)": float(100 * (code_ok & id_ok).mean())}
        if mp is not None:
            tau = TAU.get(cfg, 0.01)
            row["Accepted: code + identity + tamper map (%)"] = float(100 * (code_ok & id_ok & (mp.mean((1, 2)) <= tau)).mean())
        ID_ROWS.append(row)
    print(cfg, "identity threshold", round(theta, 3))
T_ID = pd.DataFrame(ID_ROWS)
T_ID.to_csv(f"{OUTDIR}/add_identity.csv", index=False)
print(T_ID.round(2).to_string(index=False))

## Step 13 — Summary file (send this back)

In [ ]:
def md(df):
    try:
        return df.round(4).to_markdown(index=False)
    except Exception:
        return "```\n" + df.round(4).to_string(index=False) + "\n```"
with open(f"{OUTDIR}/add_summary.md", "w") as f:
    f.write("# Additional experiments (journal)\n\n")
    f.write(f"Faces: operating point {len(EV)} | geometric {len(GI)} | identity {len(II)} | search candidates {len(CANDS)}\n\n")
    f.write("## Fixed operating point\n\n" + md(T_OP) + "\n\n")
    f.write("## Geometric attacks with search\n\n" + md(T_GEO) + "\n\n")
    f.write("## Identity-bound verification\n\n" + md(T_ID) + "\n\n")
print(open(f"{OUTDIR}/add_summary.md").read())
shutil.rmtree(MODEL_DIR, ignore_errors=True)
shutil.rmtree(VS_DIR, ignore_errors=True)